# AICE Professional 주피터라이트 실전 모의고사 02회 — 군 PBL 합성데이터(예산 집행 지연 · 군수 품목명 · 화생방 확산 방향)

## [ 필독 ] AICE Professional 모의고사 유의사항

- 공식 샘플문항(가공식품 카테고리 분류)과 **같은 시험지 형식**으로 만든 모의고사입니다. 3문항, 180분 기준입니다.
- 공식 시험과 같은 **Tabular / Text / Image 3유형**입니다. 2번 Text 문항은 공식 샘플(상품명 → 카테고리)과 같은 텍스트 분류입니다.
- 본인 스스로 모델을 만들고 **달성 목표 이상의 성능**이 나오도록 개발해야 합니다. 정답(테스트 라벨)은 제공되지 않습니다.
  대신 이 노트북 **맨 마지막 셀**(전체 자동 채점, 정답 키 인코딩)로 제출 파일을 스스로 채점할 수 있습니다(연습용).

## 사용할 수 있는 라이브러리 (주피터라이트 판)

이 모의고사는 **브라우저 주피터라이트(Pyodide · Python 3.14)** 에서 설치 없이 끝까지 풀 수 있도록 라이브러리를 제한했습니다.
필요한 라이브러리는 주피터라이트에 **미리 설치**되어 있습니다. 맨 위 '브라우저 실습' 셀은 확인만 하고 바로 끝납니다(Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다).

| 구분 | 쓸 수 있음 | 쓰지 않음(브라우저에서 안 됨) |
|---|---|---|
| 데이터·그림 | numpy · pandas · matplotlib · seaborn · Pillow | — |
| 머신러닝 | scikit-learn · LightGBM · XGBoost · imbalanced-learn · joblib | CatBoost |
| 텍스트 | scikit-learn 의 CountVectorizer · TfidfVectorizer · 내장 영어 불용어, **NLTK**(불용어·토큰화·어간 추출, 데이터는 `/drive/nltk_data` 에 미리 들어 있음) | 형태소 분석기 · BERT 등 사전학습 언어 모델 |
| 딥러닝 | `tensorflow.keras` 문법(TensorFlow.js 로 실행되는 대체 구현체, 미리 설치): `Sequential`·함수형(`Input` → 층 → `Model`) · Dense · Dropout · BatchNormalization · Conv2D · MaxPooling2D · GlobalAveragePooling2D · Embedding · LSTM · GRU · 콜백 3종(EarlyStopping · ModelCheckpoint · ReduceLROnPlateau) · `load_img` · `img_to_array` · `image_dataset_from_directory` · `ImageDataGenerator` · `model.save` / `load_model` · **MobileNet v1 전이학습**(`alpha=0.25`, 224×224, `weights='imagenet'` 만) | PyTorch · 다른 사전학습 모델(MobileNetV2/V3 · ResNet · EfficientNet 등) · 사용자 정의 층·손실 · GPU |

- 이미지는 64×64 정도로 줄여 작은 CNN 을 직접 학습하거나, MobileNet v1(alpha 0.25)로 특징을 뽑아 분류기를 학습하세요(전이학습). 브라우저 TensorFlow.js 는 CPython TensorFlow 와 수치가 조금 달라 딥러닝 점수가 약간 다르게 나올 수 있습니다.
- 학습이 오래 걸리면 epoch 를 줄이거나 조기 종료(EarlyStopping)를 쓰세요. 최신 Chrome·Edge 를 권장합니다.

## 시간 배분(180분)

| 문항 | 권장 시간 | 먼저 할 일 |
|---|---|---|
| 1. Tabular (30점) | 50분 | `info()`·`describe()` 로 자료형·결측 확인 → 기본 모델로 제출 파일부터 만들기 |
| 2. Text (35점) | 60분 | 라벨 분포·문장 길이 확인 → TF-IDF + 선형 모델로 기준 점수 |
| 3. Image (35점) | 60분 | 압축 풀기·폴더 구조 확인 → 작은 CNN 으로 기준 점수 |
| 마무리 | 10분 | 제출 파일 6개 이름·열 구조 확인 → 맨 마지막 채점 셀 실행 |

## 시험 자원 가이드

| 구분 | 공식 시험 | 이 모의고사 |
|---|---|---|
| IDE | CPU 8 core · Memory 32G · GPU 1개(VRAM 16GB) | **브라우저 주피터라이트**(설치 없음, CPU) — Colab·로컬 Jupyter 에서도 그대로 실행됨 |
| SW | Python 3.10 · TensorFlow 2.13 · PyTorch 2.1 | Python 3.14(Pyodide) · scikit-learn · LightGBM · XGBoost · Keras 문법(TensorFlow.js) — 위 표의 라이브러리만 |

## 유의 사항

- 막히면 문항 아래 접혀 있는 **힌트 1·2·3단계**를 순서대로 펼쳐 보세요. 단계가 올라갈수록 정답에 가까워집니다.
- 제출 파일명은 **고정**입니다: `answer_1~3` 모델 파일과 `answer_1~3.csv`.


문제에 제시되는 가이드는 필수적으로 준수하여 주시기 바랍니다. 이를 준수하지 않을 경우 채점 제한 또는 0점 처리될 수 있으니
각별히 유의해 주시기 바랍니다.

#### **[학습 정보]**
- 학습 목표 : Tabular · Text · Image 예측 모델 3종을 만들어 달성 목표 이상의 예측 파일을 제출하고, 제출한 코드로 같은 모델이 재현되게 할 수 있다
- AICE 역량 : PT1·PT2 표 데이터 / PX1·PX2 텍스트 / PI1·PI2 이미지 / PS1 제출 파일 형식·재현성
- 제출 파일 : `answer_1~3` 모델 파일(.pkl · .h5 등)과 `answer_1~3.csv` — **파일명은 고정**입니다(이름을 바꾸면 채점되지 않음)

| 문항 | 문제 | 달성 목표 | 역량 | 기술 | 예상 시간 |
|---|---|---|---|---|---|
| 1. Tabular | 예산 집행 지연 예측 | F1-score 0.85 이상 | PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | pandas, scikit-learn(StratifiedKFold), LightGBM, joblib | 50분 |
| 2. Text | 군수 품목명 → 품류 분류(8종) | 정확도 0.9 이상 | PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | pandas, scikit-learn(TfidfVectorizer, MultinomialNB), joblib | 60분 |
| 3. Image | 화생방 오염 확산 방향 분류 | 정확도 0.93 이상 | PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | pandas, scikit-learn(ImageDataGenerator), Keras(TensorFlow.js) | 60분 |

In [1]:
# (브라우저 실습) 주피터라이트에는 tensorflow(TensorFlow.js 대체 구현체)·scikit-learn·LightGBM 등이 미리 설치되어 있어 이 셀은 확인만 합니다.
# Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다.
import os, sys, importlib.util
if 'pyodide' in sys.modules:
    os.environ['AICE_IMG_DIR'] = '/tmp/aice_images/' + os.path.basename(os.getcwd())   # 이미지 압축은 메모리 폴더에 풂(빠름)
    if not ('tensorflow' in sys.modules or importlib.util.find_spec('tensorflow')):
        # 미리 설치되지 않은 사이트(패치 전 주피터라이트)에서만: 같은 폴더의 tfjs_keras.py 를 불러옵니다
        sys.path[:0] = [os.getcwd(), '/drive']
        import tfjs_keras
        await tfjs_keras.load()
print('준비 완료 — 다음 셀로 넘어가세요.')

준비 완료 — 다음 셀로 넘어가세요.


## 1. Tabular 문제 - 예산 집행 지연 예측 (30점)

<sub>학습 목표: 예산 집행 지연 예측 모델로 테스트 F1-score 0.85 이상을 달성할 수 있다 | 역량: PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | 기술: pandas, scikit-learn(StratifiedKFold), LightGBM, joblib | 난이도: 중급 | 예상 시간: 50분</sub>

### 문제 개요

제대·사업별 예산 집행 기록(편성액·현액·집행액·잔액·집행률·계약 방식·업체 유형·거리·평점·납품 지연일수)으로 그 건의 **집행이 지연될지** 를 예측하는 이진 분류 문제입니다(예산 집행 추적·조기 경보).

제공된 훈련 데이터셋을 활용해 이진 분류 모델을 개발하고, 테스트 데이터셋의 '집행지연여부' 값을 예측하시기 바랍니다.

### 달성 목표

"01_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **F1-score 0.85 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `01_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `01_test_x.csv`: 테스트 데이터셋입니다. 라벨('집행지연여부') 값이 제외되어 있습니다.

**Label 정보**

- '집행지연여부' : 1 = 지연, 0 = 정상 (약 50%)

**Data 정보**

- 'id' : 집행번호
- '집행기준일 · 회계연도 · 분기' : 기준일·연도·분기
- '제대코드 · 사업코드 · 사업분류' : 제대·사업·분류
- '예산편성액 · 예산현액 · 집행액 · 집행잔액 · 집행률퍼센트' : 금액(원)·집행률(%)
- '계약방식 · 업체유형 · 업체거리km · 업체평점 · 납품지연일수' : 계약·업체 정보

**데이터 출처** — 제7회 데이터한마당 본선 14팀 예산 집행 **합성데이터** 1,200건 — 실제 부대 자료 아님(정답을 드러내는 담당자 비고 열은 뺌)

### 데이터 예시

`01_train.csv`

| id | 집행기준일 | 제대코드 | 사업코드 | 사업분류 | 회계연도 | 분기 | 예산편성액 | … |
|---|---|---|---|---|---|---|---|---|
| EXE-00579 | 2024-03-19 | 제대D | 사업05 | 수리 | 2024 | 1분기 | 19470000 | … |
| EXE-00440 | 2024-12-26 | 제대I | 사업05 | 정비 | 2024 | 4분기 | 15500000 | … |
| EXE-00253 | 2024-06-23 | 제대J | 사업13 | 교육훈련 | 2024 | 2분기 | 11790000 | … |

`01_test_x.csv`

| id | 집행기준일 | 제대코드 | 사업코드 | 사업분류 | 회계연도 | 분기 | 예산편성액 | … |
|---|---|---|---|---|---|---|---|---|
| EXE-00405 | 2025-10-10 | 제대E | 사업03 | 교육훈련 | 2025 | 4분기 | 17290000 | … |
| EXE-00581 | 2025-06-15 | 제대C | 사업13 | 시설 | 2025 | 2분기 | 8720000 | … |
| EXE-01162 | 2024-11-04 | 제대D | 사업14 | 물자 | 2024 | 4분기 | 16070000 | … |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 예산 집행 지연 예측 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_1.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('집행지연여부') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_1.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 '집행지연여부' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 '집행지연여부' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 금액 열끼리의 비율(예: 집행잔액 ÷ 예산현액)이나 날짜(월) 같은 파생 변수를 만들어 보세요.
- `Pipeline` 없이 `get_dummies` 를 쓰면 테스트 열을 학습 열에 맞춰야(`reindex`) 합니다.
- 평가 지표는 **F1-score(지연=1 기준)** 입니다. 예측값은 0/1 정수로 제출하세요.

### 채점 진행 방식 안내

- 1번 문항 배점 : 30점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_1.csv 로 제출된 예측 데이터를 정답지와 대조하여, **F1-score 0.765 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [18]:
import pandas as pd
import joblib
import os
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import f1_score

# 1. 데이터 불러오기
train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
train.info(verbose=False)

LABEL = '집행지연여부'

# 2. 전처리 함수
def feats(df):
    X = df.drop(columns=['id', LABEL], errors='ignore').copy()
    X['집행월'] = pd.to_datetime(X.pop('집행기준일')).dt.month
    X['잔액비율'] = X['집행잔액'] / X['예산현액']
    X['편성대비집행비율'] = X['집행액'] / X['예산편성액']
    return pd.get_dummies(X, dtype=int)

X_train = feats(train)
X_test = feats(test)
y_train = train[LABEL]

# 3. 5-Fold 교차검증
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_1 = LGBMClassifier(
    n_estimators=100,
    random_state=42,
    verbose=-1
)

cv_pred = cross_val_predict(model_1, X_train, y_train, cv=cv)
print('기본 F1-score:', f1_score(y_train, cv_pred))

# 4. 분류 임계값 비교
cv_proba = cross_val_predict(
    model_1, X_train, y_train,
    cv=cv, method='predict_proba'
)[:, 1]

for threshold in [0.4, 0.45, 0.5, 0.55, 0.6]:
    pred = (cv_proba >= threshold).astype(int)
    print(threshold, f1_score(y_train, pred))

best_threshold = 0.4

# 5. 검증 데이터 분리
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# 6. 조기 종료 모델 학습
model_valid = LGBMClassifier(
    n_estimators=500,
    random_state=42,
    verbose=-1
)

model_valid.fit(
    X_fit, y_fit,
    eval_set=[(X_valid, y_valid)],
    eval_metric='binary_logloss',
    callbacks=[early_stopping(stopping_rounds=30, verbose=False)]
)

print('최적 학습 횟수:', model_valid.best_iteration_)

valid_proba = model_valid.predict_proba(X_valid)[:, 1]
valid_pred = (valid_proba >= best_threshold).astype(int)
print('검증 F1-score:', f1_score(y_valid, valid_pred))

# 7. 최종 모델 전체 학습
final_model = LGBMClassifier(
    n_estimators=100,
    random_state=42,
    verbose=-1
)

final_model.fit(X_train, y_train)

# 8. 모델 저장
joblib.dump(final_model, 'answer_1.pkl')

# 9. 테스트 데이터 예측
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

# 10. 예측 CSV 저장
answer_1 = test.copy()
answer_1[LABEL] = test_pred
answer_1.to_csv('answer_1.csv', index=False)

# 11. 최종 파일 확인
print('모델 파일:', os.path.exists('answer_1.pkl'))
print('CSV 파일:', os.path.exists('answer_1.csv'))
print('제출 데이터 크기:', answer_1.shape)
print('예측값 종류:', sorted(answer_1[LABEL].unique()))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Columns: 18 entries, id to 집행지연여부
dtypes: float64(3), int64(7), object(8)
memory usage: 135.1+ KB
기본 F1-score: 0.8725388601036269
0.4 0.8803245436105477
0.45 0.8784473953013279
0.5 0.8725388601036269
0.55 0.8723849372384938
0.6 0.8760504201680672
최적 학습 횟수: 34
검증 F1-score: 0.8585365853658536
모델 파일: True
CSV 파일: True
제출 데이터 크기: (240, 18)
예측값 종류: [np.int64(0), np.int64(1)]


<details><summary>힌트 1단계 (접근 방향) - 1번 문항</summary>

자료형·결측·양성(1) 비율부터 확인합니다. 번호·날짜 원문·정답과 같은 정보(누수) 열은 빼고, 범주형은 원-핫 인코딩합니다. '참고'의 파생 변수를 만든 뒤 후보 모델 두세 개를 **교차검증 F1** 로 비교하고, 양성이 적으면 **분류 임계값**도 교차검증으로 고르세요.

- 이 문항의 핵심: 금액 열끼리의 비율(예: 집행잔액 ÷ 예산현액)이나 날짜(월) 같은 파생 변수를 만들어 보세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 1번 문항</summary>

사용할 도구: `pandas`, `joblib`, `LGBMClassifier`, `StratifiedKFold`, `cross_val_predict`, `f1_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 1번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from lightgbm import ____
from sklearn.model_selection import ____, ____
from sklearn.metrics import ____

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
train.info(verbose=False)
LABEL = '____'

def feats(df):
    X = df.drop(columns=['____', LABEL], errors='____').copy()
    X['____'] = pd.to_datetime(X.pop('____')).dt.month
    X['____'] = X['____'] / X['____']
    X['____'] = X['____'] / X['____']
    return pd.get_dummies(X, dtype=int)

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 분류 임계값을 교차검증으로 고르기
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_1.pkl`(모델) · `answer_1.csv`(예측)

</details>

## 2. Text 문제 - 군수 품목명 → 품류 분류(8종) (35점)

<sub>학습 목표: 군수 품목명 → 품류 분류(8종) 모델로 테스트 정확도 0.9 이상을 달성할 수 있다 | 역량: PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | 기술: pandas, scikit-learn(TfidfVectorizer, MultinomialNB), joblib | 난이도: 중급 | 예상 시간: 60분</sub>

### 문제 개요

보급 청구서의 **품목명** 한 줄(예: `[군납] 동계용 방한 장갑 10켤레`)만 보고 **품류 8종** 을 분류하는 문제입니다. 공식 샘플문항(가공식품 상품명 → 카테고리 8종)과 같은 형식입니다.

제공된 훈련 데이터셋을 활용해 텍스트 다중 분류(8종) 모델을 개발하고, 테스트 데이터셋의 'category' 값을 예측하시기 바랍니다.

### 달성 목표

"02_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 90% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `02_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `02_test_x.csv`: 테스트 데이터셋입니다. 라벨('category') 값이 제외되어 있습니다.

**Label 정보**

- 'category' : 급식류 / 피복류 / 유류·윤활유 / 의무용품 / 통신·전자 / 수리부속 / 위생·생활 / 사무용품

**Data 정보**

- 'id' : 테스트 행 번호(테스트에만)
- 'item_name' : 품목명(수식어·규격·단위·재고번호가 섞여 있음)

**데이터 출처** — 합성 데이터 — 군 보급 품목명 표기 방식(수식어·규격·단위·재고번호)을 흉내 내 만든 6,000여 건(실제 품목 목록 아님) — 본선 13팀(품목제원관리) 형식의 **합성 품목명**

### 데이터 예시

`02_train.csv`

| item_name | category |
|---|---|
| 리필 유압유 4ml×2 | 유류·윤활유 |
| 5갤런 조달 동계용 등유 (4574-01-727-7818) | 유류·윤활유 |
| 10세트 [긴급] USB-C 무전기 배터리 | 통신·전자 |

`02_test_x.csv`

| id | item_name |
|---|---|
| 1 | 100 ml×24 조달 보온 항공유 JP-8 (9903-01-336-8763) |
| 2 | 조달 고무 살충제 100L |
| 3 | [긴급] 칫솔 100ml (5814-01-877-6709) |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 군수 품목명 → 품류 분류(8종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_2.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('category') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_2.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'category' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'category' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- `350g`·`10켤레`·`(8415-01-...)` 같은 **숫자·단위·재고번호** 는 품류와 거의 관계가 없습니다. 정규식으로 지우거나 분리해 보세요.
- `[군납]`·`★특가★` 같은 꾸밈말도 모든 품류에 섞여 있습니다.
- TF-IDF + 나이브 베이즈(`MultinomialNB`)부터 시작해 보세요. 평가 지표는 **정확도(Accuracy)** 입니다.

### 채점 진행 방식 안내

- 2번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_2.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 81% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [25]:

import pandas as pd
import joblib
import re
import os
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 데이터 불러오기
train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')
print(train['category'].value_counts().to_dict())

# 2. 학습 데이터와 정답 분리
text_X = train['item_name'].fillna('')
text_y = train['category']

# 3. 학습용·검증용 데이터 분리
text_X_fit, text_X_valid, text_y_fit, text_y_valid = train_test_split(
    text_X, text_y,
    test_size=0.2,
    random_state=42,
    stratify=text_y
)

print(text_X_fit.shape)
print(text_X_valid.shape)

# 4. TF-IDF + MultinomialNB 모델 생성
text_model = make_pipeline(
    TfidfVectorizer(),
    MultinomialNB()
)

# 5. 모델 학습
text_model.fit(text_X_fit, text_y_fit)

# 6. 검증 정확도 확인
text_valid_pred = text_model.predict(text_X_valid)

print('검증 정확도:', accuracy_score(text_y_valid, text_valid_pred))

# 7. 전체 학습 데이터로 최종 학습
text_model.fit(text_X, text_y)

# 8. 최종 모델 저장
joblib.dump(text_model, 'answer_2.pkl')

print('최종 모델 저장 완료')

# 9. 테스트 데이터 예측
text_test_X = test['item_name'].fillna('')
text_test_pred = text_model.predict(text_test_X)

# 10. 제출용 CSV 생성
answer_2 = test.copy()
answer_2['category'] = text_test_pred

answer_2.to_csv('answer_2.csv', index=False)

print('예측 CSV 저장 완료')

# 11. 최종 파일 확인
print('모델 파일:', os.path.exists('answer_2.pkl'))
print('CSV 파일:', os.path.exists('answer_2.csv'))
print('제출 데이터 크기:', answer_2.shape)
print('예측 카테고리:', sorted(answer_2['category'].unique()))


{'급식류': 936, '피복류': 819, '수리부속': 649, '의무용품': 608, '위생·생활': 607, '통신·전자': 533, '유류·윤활유': 530, '사무용품': 437}
(4095,)
(1024,)
검증 정확도: 0.974609375
최종 모델 저장 완료
예측 CSV 저장 완료
모델 파일: True
CSV 파일: True
제출 데이터 크기: (1280, 3)
예측 카테고리: ['급식류', '사무용품', '수리부속', '위생·생활', '유류·윤활유', '의무용품', '통신·전자', '피복류']


<details><summary>힌트 1단계 (접근 방향) - 2번 문항</summary>

라벨 분포와 문장 길이를 확인합니다. 문장을 TF-IDF 로 바꾸되 **단어 단위와 글자 조각(char n-gram)** 을 비교하고, 나이브 베이즈·선형 SVM·로지스틱 회귀 중 **교차검증 정확도** 가 높은 조합으로 전체 학습하세요. 다른 컬럼(숫자·범주)이 단서이면 `ColumnTransformer` 로 함께 넣습니다.

- 이 문항의 핵심: `350g`·`10켤레`·`(8415-01-...)` 같은 **숫자·단위·재고번호** 는 품류와 거의 관계가 없습니다. 정규식으로 지우거나 분리해 보세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 2번 문항</summary>

사용할 도구: `pandas`, `joblib`, `make_pipeline`, `TfidfVectorizer`, `MultinomialNB`, `train_test_split`, `accuracy_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 2번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
from sklearn.pipeline import ____
from sklearn.feature_extraction.text import ____
from sklearn.naive_bayes import ____
from sklearn.model_selection import ____
from sklearn.metrics import ____

train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')
print(train['____'].value_counts().to_dict())


# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_2.pkl`(모델) · `answer_2.csv`(예측)

</details>

## 3. Image 문제 - 화생방 오염 확산 방향 분류 (35점)

<sub>학습 목표: 화생방 오염 확산 방향 분류 모델로 테스트 정확도 0.93 이상을 달성할 수 있다 | 역량: PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | 기술: pandas, scikit-learn(ImageDataGenerator), Keras(TensorFlow.js) | 난이도: 중급 | 예상 시간: 60분</sub>

### 문제 개요

지형 위에 그린 **화생방 오염 확산도**(48×48 컬러, 검은 십자 = 오염원)로 오염이 퍼져 나가는 **방향 8종**(북·북동·…·북서)을 분류하는 문제입니다. 확산 폭·길이·농도가 다르고, 다른 곳의 작은 오염 반점과 구름 가림이 섞여 있습니다.

제공된 훈련 데이터셋을 활용해 이미지 다중 분류(8종) 모델을 개발하고, 테스트 데이터셋의 'label' 값을 예측하시기 바랍니다.

### 달성 목표

"03_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 93% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `03_images.zip`: 압축을 풀면 `03_train/cbrn_*.png`(학습, **한 폴더**)와 `03_test/test_*.png`(테스트)가 생깁니다. 48×48 컬러 PNG.
- `03_train.csv`: 학습 이미지 파일명과 라벨
- `03_test_x.csv`: 테스트 이미지 파일명(라벨 제외)

**Label 정보**

- 'label' : N: 북 / NE: 북동 / E: 동 / SE: 남동 / S: 남 / SW: 남서 / W: 서 / NW: 북서 — 오염이 퍼져 나가는(풍하) 방향, 위쪽이 북

**데이터 출처** — v3 합성 — 본선 8팀(핵·화생방 상황 방호) 맥락의 오염 확산도 48×48 컬러, 방향마다 250장(실제 상황 아님)

### 데이터 예시

학습 이미지 라벨별 장수: S 200, W 200, N 200, NE 200, E 200, NW 200, SW 200, SE 200

| file_name | label |
|---|---|
| cbrn_00001.png | S |
| cbrn_00002.png | W |
| cbrn_00003.png | N |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 화생방 오염 확산 방향 분류 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_3.h5** (또는 다른 확장자의 모델 파일)

(2) 라벨('label') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_3.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'label' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'label' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 학습 이미지가 한 폴더에 있으므로 `flow_from_dataframe(train, directory=..., x_col='file_name', y_col='label')` 로 읽습니다.
- **방향이 곧 라벨** 입니다. 좌우·상하 뒤집기나 회전 증강을 하면 라벨이 바뀌니 쓰지 마세요.
- 테스트는 `shuffle=False` 로 읽어야 `03_test_x.csv` 순서와 맞습니다. 평가 지표는 **정확도(Accuracy)** 입니다.

### 채점 진행 방식 안내

- 3번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_3.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 83.7% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [43]:

import os
import zipfile

# 1. 이미지 압축 해제
IMG_DIR = os.environ.get('AICE_IMG_DIR', '03_images')

if not os.path.exists(os.path.join(IMG_DIR, '03_train')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)

# 2. 라이브러리
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 3. CSV 불러오기
train = pd.read_csv('03_train.csv')
test = pd.read_csv('03_test_x.csv')
SIZE = 48

# 4. 이미지 정규화
gen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2
)

# 5. 이미지 설정
kw = dict(
    directory=f'{IMG_DIR}/03_train',
    x_col='file_name',
    y_col='label',
    target_size=(SIZE, SIZE),
    class_mode='categorical',
    batch_size=32,
    seed=42
)

# 6. 학습용·검증용 이미지 불러오기
train_gen = gen.flow_from_dataframe(
    dataframe=train,
    subset='training',
    shuffle=True,
    **kw
)

valid_gen = gen.flow_from_dataframe(
    dataframe=train,
    subset='validation',
    shuffle=False,
    **kw
)

print('클래스:', train_gen.class_indices)

# 7. CNN 모델 구성
image_model = Sequential([
    Input(shape=(SIZE, SIZE, 3)),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(8, activation='softmax')
])

image_model.summary()

# 8. 모델 컴파일
image_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 9. 조기 종료 설정
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

# 10. CNN 학습
image_history = image_model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=25,
    callbacks=[early_stop, reduce_lr]
)

print('최고 검증 정확도:', max(image_history.history['val_accuracy']))

# 11. 최종 검증
valid_loss, valid_accuracy = image_model.evaluate(valid_gen)

print('최종 검증 정확도:', valid_accuracy)
print('최종 검증 손실:', valid_loss)

# 12. 모델 저장
image_model.save('answer_3.h5')

# 13. 테스트 이미지 불러오기
test_gen = ImageDataGenerator(
    rescale=1.0 / 255
).flow_from_dataframe(
    dataframe=test,
    directory=f'{IMG_DIR}/03_test',
    x_col='file_name',
    y_col=None,
    target_size=(SIZE, SIZE),
    class_mode=None,
    batch_size=32,
    shuffle=False
)

# 14. 테스트 예측
image_proba = image_model.predict(test_gen)
image_pred_idx = np.argmax(image_proba, axis=1)

# 15. 숫자를 방향 문자로 변환
idx_to_label = {v: k for k, v in train_gen.class_indices.items()}
image_pred_labels = [idx_to_label[int(i)] for i in image_pred_idx]

# 16. CSV 저장
answer_3 = test.copy()
answer_3['label'] = image_pred_labels
answer_3.to_csv('answer_3.csv', index=False)

# 17. 제출 파일 확인
print('모델 파일:', os.path.exists('answer_3.h5'))
print('CSV 파일:', os.path.exists('answer_3.csv'))
print('제출 데이터 크기:', answer_3.shape)
print('예측 방향:', sorted(answer_3['label'].unique()))

Found 1280 validated image filenames belonging to 8 classes.
Found 320 validated image filenames belonging to 8 classes.
클래스: {'E': 0, 'N': 1, 'NE': 2, 'NW': 3, 'S': 4, 'SE': 5, 'SW': 6, 'W': 7}


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 46, 46, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       131,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,904 (878.53 KB)

 Trainable params: 224,904 (878.53 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 19s 312ms/step - accuracy: 0.1766 - loss: 2.0372 - val_accuracy: 0.3406 - val_loss: 1.7748 - learning_rate: 0.0010
Epoch 2/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 16s 204ms/step - accuracy: 0.4617 - loss: 1.3336 - val_accuracy: 0.6844 - val_loss: 0.8405 - learning_rate: 0.0010
Epoch 3/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 158ms/step - accuracy: 0.6375 - loss: 0.9075 - val_accuracy: 0.7531 - val_loss: 0.5975 - learning_rate: 0.0010
Epoch 4/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 197ms/step - accuracy: 0.7258 - loss: 0.7182 - val_accuracy: 0.8719 - val_loss: 0.3982 - learning_rate: 0.0010
Epoch 5/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 155ms/step - accuracy: 0.7820 - loss: 0.5691 - val_accuracy: 0.9031 - val_loss: 0.2794 - learning_rate: 0.0010
Epoch 6/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 198ms/step - accuracy: 0.8375 - loss: 0.4336 - val_accuracy: 0.8969 - val_loss: 0.2645 - learning_rate: 0.0010
Epoch 7/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 157ms/step - accuracy: 0.8992 - loss: 0.2936 

최종 검증 정확도: 0.987500011920929
최종 검증 손실: 0.026957755908370018
Found 400 validated image filenames.
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step
모델 파일: True
CSV 파일: True
제출 데이터 크기: (400, 2)
예측 방향: ['E', 'N', 'NE', 'NW', 'S', 'SE', 'SW', 'W']


<details><summary>힌트 1단계 (접근 방향) - 3번 문항</summary>

압축을 풀고 폴더 구조(학습: 클래스별 폴더, 테스트: 한 폴더)를 확인합니다. 이미지를 같은 크기로 읽어 0~1 로 맞추고, 검증 데이터를 떼어 **작은 CNN**(또는 사전학습 모델 특징 + 분류기)을 학습합니다. 증강·Dropout·조기 종료로 과적합을 막고, 테스트는 `03_test_x.csv` 순서대로 예측하세요.

- 이 문항의 핵심: 학습 이미지가 한 폴더에 있으므로 `flow_from_dataframe(train, directory=..., x_col='file_name', y_col='label')` 로 읽습니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 3번 문항</summary>

사용할 도구: `tensorflow`, `pandas`, `ImageDataGenerator`, `Sequential`, `load_model`, `Input`, `Conv2D`, `MaxPooling2D`, `Flatten`, `Dense`, `Dropout`, `EarlyStopping`, `ReduceLROnPlateau`.

- 참고 문서: <https://keras.io/api/models/sequential/>
- 참고 문서: <https://keras.io/api/layers/convolution_layers/convolution2d/>
- 참고 문서: <https://keras.io/api/layers/pooling_layers/max_pooling2d/>
- 참고 문서: <https://keras.io/api/layers/core_layers/dense/>
- 참고 문서: <https://keras.io/api/layers/regularization_layers/dropout/>
- 참고 문서: <https://keras.io/api/callbacks/early_stopping/>

</details>

<details><summary>힌트 3단계 (코드 구조) - 3번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
IMG_DIR = os.environ.get('____', '____')        # 압축을 풀 위치(브라우저판은 메모리 폴더 /tmp)
if not os.path.exists(os.path.join(IMG_DIR, '____')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)
import ____
from tensorflow.keras.preprocessing.image import ____
from tensorflow.keras.models import ____, ____
from tensorflow.keras.layers import ____, ____, ____, ____, ____, ____
from tensorflow.keras.callbacks import ____, ____

train = pd.read_csv('03_train.csv')
test = pd.read_csv('03_test_x.csv')
SIZE = ____
gen = ImageDataGenerator(rescale=____ / ____, validation_split=____)        # 방향이 라벨이라 뒤집기·회전 증강은 쓰지 않음
kw = dict(directory=f'____', x_col='____', y_col='____', target_size=(SIZE, SIZE), class_mode='____', batch_size=____, seed=____)

# ▷ 모델 구성(CNN 또는 사전학습 특징 추출)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_3.h5`(모델) · `answer_3.csv`(예측)

</details>

## 제출할 파일 목록

| 문항 | 모델 파일 | 예측 파일 |
|---|---|---|
| 1. Tabular | answer_1.pkl (다른 확장자 가능) | answer_1.csv |
| 2. Text | answer_2.pkl (다른 확장자 가능) | answer_2.csv |
| 3. Image | answer_3.h5 (다른 확장자 가능) | answer_3.csv |
| 공통 | 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장) | |

(주의) 제출 파일명은 **고정**입니다. 위 표의 이름 그대로 저장하세요(앞부분 `answer_` · 문항 번호 · 확장자).
**예시 : answer_1.pkl , answer_1.csv** (이름을 바꾸거나 공백·대쉬(-)를 넣으면 채점되지 않습니다)

## 안내사항

- 이 모의고사는 브라우저 주피터라이트에서 미리 설치된 라이브러리만 씁니다(맨 위 '사용할 수 있는 라이브러리' 표).
- 모델 저장시 본인이 사용하는 프레임워크의 모델 저장 확장자를 사용하셔도 됩니다.(h5, keras, pkl, pickle 등)

수고하셨습니다.

In [45]:
# 전체 자동 채점 — 세 문항의 제출 파일(answer_1~3.csv·모델 파일)을 만든 뒤 이 셀을 실행하세요.
# 정답(테스트 라벨)은 압축·인코딩되어 있어 보이지 않습니다. 문항별 지표·통과 여부와 총점을 출력합니다.
PREFIX = 'answer'                                  # 제출 파일명 앞부분(고정)

import base64, json, os, re, zlib
import numpy as np
import pandas as pd

_KEY = "IoDvBhExRh1KpHE8lZyWAGIsKMWxBrxq3XqeIOlseCBCM0l41pqenILCGkgY9l6TssqeXi5SfhQoknNlfUCn3bb23iz84P3koPCdzc+8i4ePrySvZkRngIHn5W+BhEUnLrm5nfnHh5+Fq5dB+R3hwWGpvLwA7OKAwIbFN3d0HK+BNXTkJQPlJwjlpPckJyykvcmLuKusNuTtYzfEZUJtjamDpCa5m7yM9uPP123MjQWGwLQlhGanLizqa4cHFO2jk6ivk4SZgyYrx4fgLf0DIejl5evv42kBhOChGym1+ts1zm/1dc1KpT+w54LqghWMLXWxIXVp9My78y7aaE/6a+m+Aoc/PIbMT59OXqAMadxEk55BQTMzP/TrUP4UVXFh1Cdh+Pr4Is2YZ5WEDIuSQjRAaa4PyuIkIqmczxznh4vyLYBAL2DDlUS0tEcur7I+YKc2NWhnUu+zNxjBJEJyiSG5MQcARcXEpSautQWvqxVNZUZ19MCbr6vpoLOTuLyJyZ+XosfRtaRjpaeHODYpq4WFdbENfQ1NTZWFj/UUTbXEZPGFJpfU9BXUlJWk8iWmr7iBTc05eVosHeZiYKX/axivqfUgjKWoVpVDdtb9X3l4gVS9yvHuS4nj2vHDbEMIDBaBB8k/Xt/0ytR7KODVQ9ypOW+CxYIf2krbLJqWmypPCOJr1jjXeErCOqekg2p+2i925twixD5RE8QkJJn62v9bcJ849pAZLr9PD11E1x1r6vAeGpEbl9QOUvs7GC/wcVzPeEiFuoDZTg/xQ70aJnqDEvn/GyJXvAu/Gdmo2fTaGzeTBxmc6tj40M8wzA+eqbRKivCnwJCZwUyH0iuWAi753UEhVzpQck56SgLVQCdgWY14i1j4V2zw6WjOodq7XKoh2uS0DaZgBH4bvm8sorq0++96xxJ66nwrEwJadRMgGFxf9p1uXCsb/JY+cu5j4VFa9IgFHHbAsqZLKD5tRvZlVyJCsVXPv2asXV1NgNxORthCfwODG09s3nlfHtvWIBOx+ZT0uVi/DvrgLSMkdEeWYLEw8jacymfCGRvW8CsPX5RSgHoETWya/lNYuDoN8HQleYLUn8IrFF1/P9o6LDaCiVQZoj8SM8MJcj5El8lHan8o68TYLAyNkWL1P61Wy1DfaPF1VpLzkD4AbjnVri+bag479va4CELWUH05TjGqnAAplsUmthr73xSmwK5CFE3AUjYC0Pd1fv6aF6a3FMX+D5tOI5yTY12n8iKFFFkZ+EkMQOpo+2kVFKV/e1HSw0uhzgIqc8XN8dHiaL9ms5op/E5jKkTP3kbDJQ3HGq9Dkjp0eye4VqZW7HN3CntoTPzywateLufdw9mWDf2Pi7Z7YgNEPVhAaD0fSsEphrM8eejeufe/q2NF4GqTvZPKpZF/DAbMyBJDf+5IQ31oBGrAgDt9Rs8i7KeT+Os25Epz8R6DQB26nA7dYeLsPbaHn597ppHOTOutwxl7v5qAYTq4DuHuAsc+IVXet9jzPYM56P6kZSUp23sLGC8RAy/6+eI2sdTq7YQiyTLq+PbBTxEZ++EBaf9nl8TKfQEK8sDVe0m7u1G2ksgjJRzfyOjX7616JjDUs8P6xxbtSSfhFjNCnMM2yVI8zM5ikvXl87DGfRogRVcHc3cyzkSoIHuWw0xw/C0BHYritzHbnqMDKO3WL4urPmwfJsnBd/fCgg3VC5NupGhUw+Xnwql+FkacMiENU3hzgVYnXqzeWZ95HPGUze5GpOMFKoc3lgYEj4dNbArBNhAVbc8TE1FckuzU+0X9xpC7D/yb1RBk2xBqQDNVtkNw6cU0Mv3vzdIX5rebbWS3EA1xiQhzDu+R86eGYwoMPGYczrZfeYi8khc/oKh2V02kMFhPnmhEQkdlg4Ehx1MPmXLoEhlcwDbpe0ftCh864UxF4knFDS7ciUpZ3qKzF4uWK5KeaRqGg23o9JuigyR1yO0tdOmwLWJaFQsdAgmIqaxsr7563aDkGqiNQOJEQ6QSRmZfG8UYfWy9E5PL6MOI6DUResvdBEZ9vAOjL6IvPIEcxo+txpx65ea4oaCXSr0aR0VQuCZJDS34A8vh1A1UiWO19n9+rtCgyfj0IPhnyRDhthPck8tO+1Rn8lNfJqCMRoFiB3+kYR6u6yVg4nuB34lAnIJM7I41I38mxZEjuGzvyhVdrTZoQLrur0Ob76SVUgSkOabcugClbV7BLWwsfIFpo3Ob76oB2DEt9WV8Ar6XoT+PpnSl5+CLoOUyJPCxxbigj5HNkLHB8X0frZswqr/KMO88IePorxBLZjeHtFbP+k23TX/GP4Tf7KcAe/xxDcHy2AVbC0saDuxqDF9K0/MRazNPhRSLGOLgkYIUbqrxgqRRP9gGGocVvGKDAXlZiM9IwEoLdza+WCSrfnoqmlFb2luA+dJ3fxvY6yM24ePKy0qSj84PTenfPqfMIkzP2mtNDnAZFAYbjNlSe7JIkw+1wbd/2UH/WuBSLA4ss8DR69u0kzmBS4Ad+O/OElNiegL8CO8HS1Ix1oj/jgM3+A1rTl7Sc/BCUptuW+kOa/YQIvEkxwrQU/4aUk+K+fK/AD9ETdkIMP1owmrVMVLrg3i6a+P15pFpI8bAKLkjdQ1PZ2oQIpFC3ZGfWAAuzxIyMf6avEs0c3F5HVzMCWJePy45W1bbQ1XWOYNrUrHdsd2lWeKQA/w="


def _key():
    raw = base64.b64decode(_KEY)
    return json.loads(zlib.decompress(bytes(b ^ 0x5A for b in raw)).decode("utf-8"))


def _metric(m, y, p):
    if m == "rmse":
        return float(np.sqrt(np.mean((np.asarray(p, float) - np.asarray(y, float)) ** 2)))
    y, p = np.asarray(y).astype(str), np.asarray(p).astype(str)
    if m == "accuracy":
        return float((y == p).mean())
    labels = sorted(set(y)) if m == "macro_f1" else ["1"]
    f1s = []
    for c in labels:
        tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c)); fn = np.sum((p != c) & (y == c))
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s))


def grade(prefix="answer", folder="."):
    key = _key()
    phone = prefix                                      # 제출 파일명 앞부분(고정: answer_N.csv · answer_N.pkl/.h5)
    total, rows = 0, []
    for q in key["problems"]:
        no, pts = q["no"], q["points"]
        csv = os.path.join(folder, f"{phone}_{no}.csv")
        models = [f for f in os.listdir(folder) if f.startswith(f"{phone}_{no}.") and not f.endswith(".csv")]
        msg, score, val = [], 0, None
        if not models:
            msg.append("모델 파일 없음")
        if not os.path.exists(csv):
            rows.append((no, pts, 0, None, "예측 CSV 없음")); continue
        sub = pd.read_csv(csv)
        test = pd.read_csv(os.path.join(folder, q["test_file"]))
        lab = q["label"]
        if list(sub.columns) != list(test.columns) + [lab]:
            msg.append(f"열 구조가 다름(테스트 열 + 마지막 '{lab}' 열이어야 함)")
        if lab not in sub.columns or len(sub) != len(test):
            rows.append((no, pts, 0, None, f"행 수 {len(sub)} ≠ {len(test)} 또는 '{lab}' 열 없음")); continue
        keycol = q["key_col"]
        if keycol and not (sub[keycol].astype(str).values == test[keycol].astype(str).values).all():
            msg.append(f"'{keycol}' 순서가 테스트 파일과 다름")
        if sub[lab].isna().any():
            msg.append("예측값에 결측 있음"); sub[lab] = sub[lab].fillna(0 if q["metric"] == "rmse" else "")
        val = _metric(q["metric"], q["y"], sub[lab].values)
        ok = val <= q["pass_line"] if q["metric"] == "rmse" else val >= q["pass_line"]
        if ok and not msg:
            score = pts
        rows.append((no, pts, score, val, "; ".join(msg) or ("통과" if ok else "기준선 미달")))
        total += score
    print(f"{'문항':<6}{'배점':>4}{'점수':>6}  {'지표':<10}{'결과':>10}  {'목표':>8}  {'기준선':>8}  비고")
    for (no, pts, sc, val, note), q in zip(rows, key["problems"]):
        v = "-" if val is None else (f"{val:.4f}")
        print(f"{no:<6}{pts:>4}{sc:>6}  {q['metric']:<10}{v:>10}  {q['target']:>8}  {q['pass_line']:>8}  {note}")
    print(f"총점: {total} / 100  ({'합격' if total >= 80 else '불합격'}, 합격 기준 80점)")
    return total


_점수 = grade(prefix=PREFIX)

문항      배점    점수  지표                결과        목표       기준선  비고
1       30    30  f1            0.8797      0.85     0.765  통과
2       35    35  accuracy      0.9703       0.9      0.81  통과
3       35    35  accuracy      0.9725      0.93     0.837  통과
총점: 100 / 100  (합격, 합격 기준 80점)
